# 04. Pydantic による型安全な構造化出力（response_schema）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/04_structured_outputs.ipynb)

プロンプトだけで「JSON 形式のみで回答してください」と指示する手法は、前置き文の混入や ` ```json ` コードフェンスの付与によって本番環境で頻繁にパースエラーを引き起こします。`response_mime_type="application/json"` と `response_schema=<Pydantic BaseModel>` を併用すると **文法制約付きデコーディング（Grammar-Constrained Decoding）** が有効化され、Gemini のトークン生成自体がスキーマ文法に厳密に制約されるため、正規表現や文字列整形を一切使わずに `response.parsed` から検証済みの Pydantic インスタンスを直接取得できます。

![図4-1: Pydantic (response_schema) による制約付き構造化出力パイプライン](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_structured_pipeline_ja.png)

*図4-1: Pydantic (response_schema) による制約付き構造化出力パイプライン*

---

## 1. 3行のクライアント初期化とネストされた Pydantic スキーマ定義

まず、Google Colab のシークレット（左サイドバーの `🔑` アイコン）に保存した `GEMINI_API_KEY` を読み込み、`google-genai` クライアントを初期化します。

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

次に、非構造化の障害対応ログからポストモーテム情報を構造化抽出するため、ネストされた Pydantic スキーマ（`IncidentSummary` 内の `list[ActionItem]`）を定義します。`response_schema` に `IncidentSummary` クラスを渡すだけで、SDK が JSON Schema への変換と `response.parsed` へのデシリアライズを自動実行します。

In [ ]:
from typing import Literal

from google.genai import types
from pydantic import BaseModel, Field


class ActionItem(BaseModel):
    owner: str = Field(description="対応タスクを担当するチーム名または担当者名。")
    task: str = Field(description="具体的かつ実行可能な再発防止エンジニアリング施策。")
    priority: Literal["P0", "P1", "P2"] = Field(description="P0（最優先）から P2 までの優先度区分。")


class IncidentSummary(BaseModel):
    title: str = Field(description="40文字以内の簡潔なインシデント件名。")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(
        description="顧客影響度に基づく重大度レベル。"
    )
    root_cause: str = Field(description="技術的な根本原因を1文で要約した説明。")
    affected_services: list[str] = Field(description="障害の影響を受けたマイクロサービス名の一覧。")
    action_items: list[ActionItem] = Field(description="優先度付きの再発防止アクションアイテム一覧。")


raw_incident_log = """
[2026-04-12 14:03 UTC] PagerDuty アラート: checkout-api の 5xx エラー率が 38% に急増。
[14:09 UTC] オンコール SRE が PostgreSQL コネクションプールの使用率 100% 飽和を確認。
[14:16 UTC] 根本原因を特定: 上流の決済 Webhook タイムアウトを契機に、payment-worker から
checkout-api への無制限なリトライストームが発生していた。
[14:24 UTC] サーキットブレーカーによる負荷遮断と滞留ワーカーの再起動により復旧。
振り返りミーティングで合意した恒久対策:
1. SRE チーム: 指数バックオフの上限設定とサーキットブレーカー閾値の即時適用 (P0)
2. 決済チーム: PgBouncer プール使用率 80% 時点での飽和事前アラート追加 (P1)
3. 基盤チーム: 次スプリントの負荷試験で Webhook タイムアウト時の再現テストを実施 (P2)
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=raw_incident_log,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=IncidentSummary,
        temperature=0.1,
    ),
)

# response.parsed は IDE の型補完が効く検証済み IncidentSummary インスタンスを返す
summary: IncidentSummary = response.parsed
print(f"[{summary.severity}] {summary.title} (対策タスク: {len(summary.action_items)}件)")
print(summary.model_dump_json(indent=2))

---

## 2. スキーマ設計のベストプラクティスと Drawlib による抽出データの可視化

文法制約付きデコーディングは JSON の構文破損を100%防ぎますが、実務での **抽出精度** と **応答速度** をさらに高めるには、以下の4つの Pydantic スキーマ設計原則が有効です。

![図4-2: LLM 構造化抽出の精度を高める Pydantic スキーマ設計パターン](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_schema_best_practices_ja.png)

*図4-2: LLM 構造化抽出の精度を高める Pydantic スキーマ設計パターン*

`response_schema` 用のモデルを設計する際は、次の4点を意識してください。

1. **閉じたカテゴリには `Literal` または `Enum` を使う**: `Literal["P0", "P1", "P2"]` のように選択肢を列挙することで、表記揺れや想定外ラベルによる後続処理の分岐ミスを防止できます。
2. **任意の `dict[str, Any]` ではなく `list[SubModel]` を合成する**: キー名が不定の辞書型は文法制約を効かせにくいため、明示的な属性を持つサブモデルの配列としてモデリングします。
3. **曖昧な指標には簡潔な `Field(description=...)` を添える**: `description` は JSON Schema にそのまま埋め込まれるため、プロンプト本文を肥大化させずに文字数制限や判定基準をモデルへ伝達できます。
4. **高速・決定論的な抽出では `temperature=0.0`〜`0.2` と `thinking_budget=0` を組み合わせる**: `gemini-2.5-flash` で定型データ抽出を行う場合、`thinking_config=types.ThinkingConfig(thinking_budget=0)` を併用するとレイテンシと出力のばらつきを最小化できます。

抽出結果の `summary.action_items` はすでに型安全な `list[ActionItem]` になっているため、そのまま **Drawlib の `Table`**（`setup(..., cwd=True)` + `Table` + `show()`）へ渡すだけで、ノートブックセル内にエグゼクティブ向けの対策一覧表を描画できます。

In [ ]:
from drawlib.canvas import setup, show
from drawlib.smartarts import Table
from drawlib.styles import Colors, Styles
from drawlib.text import text

setup(width=120, height=52, cwd=True)

text(
    (60, 46.5),
    f"[{summary.severity}] {summary.title}",
    style=Styles.DarkBold.patch(text_size=11.5),
)

items = summary.action_items[:4]
rows = [["優先度", "担当", "再発防止アクション"]] + [
    [item.priority, item.owner, item.task] for item in items
]

table = Table(
    cell_style=Styles.White,
    text_style=Styles.Dark.patch(text_size=10.0),
    header_cell_style=Styles.PrimaryFlat,
    header_text_style=Styles.WhiteBold.patch(text_size=10.5),
    border_style=Styles.MutedThin,
)
table.set_style_cell_evenodd(
    even_color=Colors.Light,
    even_text_style=Styles.Dark.patch(text_size=10.0),
    odd_color=Colors.White,
    odd_text_style=Styles.Dark.patch(text_size=10.0),
)
table.set_style_cell_header(
    background_color=Colors.Primary,
    text_style=Styles.WhiteBold.patch(text_size=10.5),
)

row_heights = [9.0] + [27.0 / max(len(items), 1)] * len(items)
table.draw_flexible(
    xy=(6, 41),
    column_widths=[18, 26, 64],
    row_heights=row_heights,
    data=rows,
    align="top_left",
)

show()